In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


from xgboost import XGBRegressor
from pymongo import MongoClient
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline


In [3]:
# ==================================================
# LOAD PROCESSED DATASET
# ==================================================

import pandas as pd

df = pd.read_csv("processed_dataset.csv")

print(f"✅ Archivo cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas")
print("\nPrimeras filas:")
df.head()

✅ Archivo cargado correctamente: 66954 filas y 26 columnas

Primeras filas:


,RACEID,DRIVER_POINTS_BEFORE_RACE,POINTS,DRIVERID,DRIVERREF,LAPS,MILLISECONDS,WEATHER_rain,WEATHER_WET,SCORE,...,PS_COUNT,SC_COUNT,YEAR,ROLLING_POINTS,ROLLING_LAP,LAP_CONSISTENCY,RACE_INTERRUPTIONS,OVERTAKE_RATIO,DRIVER_ENCODED,RACE_ENCODED
0,833,NaN,-0.375795,579,fangio,62.0,7.105023e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,241,1
1,833,NaN,-0.375795,589,chiron,26.0,2.979526e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,159,1
2,833,NaN,-0.375795,619,gerard,67.0,7.678009e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,295,1
3,833,NaN,-0.316728,627,rosier,68.0,7.792606e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.316728,NaN,NaN,NaN,0.0,678,1
4,833,NaN,-0.375795,640,graffenried,36.0,4.125497e+06,NaN,NaN,5.769466,...,NaN,NaN,1950.0,-0.375795,NaN,NaN,NaN,0.0,314,1


In [ ]:
## SVM

In [4]:
def train_svm_regression_model(df):

    # ============================================
    # FEATURES & TARGET
    # ============================================

    features = [
        "POINTS",
        "LAPS",
        "MILLISECONDS",
        "WEATHER_cloudy",
        "OVERTAKEN_POSITIONS_TOTAL",
        "DNF_COUNT",
        "LAPMEAN",
        "PS_COUNT",
        "SC_COUNT",
        "DRIVER_ENCODED",
        "RACE_ENCODED"
    ]

    target = "SCORE"

    # ============================================
    # CLEAN DATA
    # ============================================

    model_df = df[features + [target]].copy()

    model_df = model_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    model_df = model_df.dropna()

    # ============================================
    # X & y
    # ============================================

    X = model_df[features]

    y = model_df[target].values.ravel()

    # ============================================
    # TRAIN TEST SPLIT
    # ============================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # ============================================
    # SVM PIPELINE
    # ============================================

    svm_model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "svr",
            SVR(
                kernel="rbf",
                C=100,
                gamma="scale",
                epsilon=0.1
            )
        )
    ])

    # ============================================
    # TRAIN
    # ============================================

    svm_model.fit(
        X_train,
        y_train
    )

    # ============================================
    # PREDICTIONS
    # ============================================

    y_pred = svm_model.predict(
        X_test
    )

    # ============================================
    # METRICS
    # ============================================

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # ============================================
    # PREDICTIONS DF
    # ============================================

    predictions_df = pd.DataFrame({
        "Actual_SCORE": y_test,
        "Predicted_SCORE": y_pred
    })

    # ============================================
    # RETURN
    # ============================================

    return {
        "model": svm_model,
        "features": features,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "predictions": predictions_df,
        "X_test": X_test,
        "y_test": y_test
    }

In [ ]:
## XGBOOST

In [5]:
def train_xgboost_regression_model(df):

    # ============================================
    # FEATURES & TARGET
    # ============================================

    features = [
        "POINTS",
        "LAPS",
        "MILLISECONDS",
        "WEATHER_cloudy",
        "OVERTAKEN_POSITIONS_TOTAL",
        "DNF_COUNT",
        "LAPMEAN",
        "PS_COUNT",
        "SC_COUNT",
        "DRIVER_ENCODED",
        "RACE_ENCODED"
    ]

    target = "SCORE"

    # ============================================
    # CLEAN DATA
    # ============================================

    model_df = df[features + [target]].copy()

    for col in model_df.columns:
        model_df[col] = pd.to_numeric(
            model_df[col],
            errors="coerce"
        )

    model_df = model_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    model_df = model_df.dropna()

    # ============================================
    # X & y
    # ============================================

    X = model_df[features]

    y = model_df[target]

    # ============================================
    # TRAIN TEST SPLIT
    # ============================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # ============================================
    # XGBOOST MODEL
    # ============================================

    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )

    # ============================================
    # TRAIN
    # ============================================

    xgb_model.fit(
        X_train,
        y_train
    )

    # ============================================
    # PREDICTIONS
    # ============================================

    y_pred = xgb_model.predict(
        X_test
    )

    # ============================================
    # METRICS
    # ============================================

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred
        )
    )

    r2 = r2_score(
        y_test,
        y_pred
    )

    # ============================================
    # FEATURE IMPORTANCE
    # ============================================

    importance_df = pd.DataFrame({
        "Feature": features,
        "Importance": xgb_model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        by="Importance",
        ascending=False
    )

    # ============================================
    # PREDICTIONS DF
    # ============================================

    predictions_df = pd.DataFrame({
        "Actual_SCORE": y_test.values,
        "Predicted_SCORE": y_pred
    })

    # ============================================
    # RETURN
    # ============================================

    return {
        "model": xgb_model,
        "features": features,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "predictions": predictions_df,
        "feature_importance": importance_df,
        "X_test": X_test,
        "y_test": y_test
    }